# GPU smoke test — semiconductor image restoration

**Before running:** menu `Runtime` → `Change runtime type` → select **T4 GPU** → Save.

Then `Runtime` → `Run all`. Cell 3 will ask you to choose a file: pick `project.zip`.

What this notebook proves:
1. the GPU is visible to PyTorch
2. our training pipeline runs on GPU end-to-end (2-epoch mini training)
3. how fast one epoch of REAL full-size training would be

In [ ]:
# 1. Is there a GPU? Should print a table with "Tesla T4".
!nvidia-smi

In [ ]:
# 2. Install the two packages Colab doesn't have (torch is preinstalled).
%pip install -q pytorch-msssim lpips

In [ ]:
# 3. Upload project.zip when the file chooser appears, then unpack it.
from google.colab import files
uploaded = files.upload()
!unzip -o -q project.zip
!ls

In [ ]:
# 4. Make a tiny fake dataset (striped test images) and run the same
#    2-epoch smoke test we ran on CPU — this time it must say device: cuda.
!python make_fakedata.py
!python train.py --mode synthetic --train-dir fakedata/train --val-dir fakedata/val --scale 4 --patch 64 --width 16 --blocks 2 --batch-size 2 --epochs 2 --val-every 1 --workers 2 --out ckpt_smoke

In [ ]:
# 5. Speed benchmark of the REAL model size (width 48, 24 blocks,
#    192px patches, batch 16): how many training steps per second?
import time, torch
from models.nafnet import NAFNetSR
from utils.losses import RestorationLoss

device = "cuda"
model = NAFNetSR(channels=1, width=48, n_blocks=24, scale=4).to(device)
loss_fn = RestorationLoss(channels=1).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=2e-4)

degraded = torch.rand(16, 1, 48, 48, device=device)   # 192/4 = 48px inputs
clean = torch.rand(16, 1, 192, 192, device=device)

for _ in range(3):                       # warm-up (GPU compiles kernels)
    loss, _ = loss_fn(model(degraded), clean)
    opt.zero_grad(); loss.backward(); opt.step()
torch.cuda.synchronize()

t0 = time.time(); steps = 20
for _ in range(steps):
    loss, _ = loss_fn(model(degraded), clean)
    opt.zero_grad(); loss.backward(); opt.step()
torch.cuda.synchronize()
per_step = (time.time() - t0) / steps

print(f"{per_step:.3f} s per training step (batch of 16 patches)")
for n_images in (100, 500, 1000):
    steps_per_epoch = n_images // 16
    total_h = 200 * steps_per_epoch * per_step / 3600
    print(f"dataset of {n_images:4d} images: "
          f"~{steps_per_epoch * per_step:5.1f} s/epoch, "
          f"~{total_h:4.1f} h for a full 200-epoch run")

## Success looks like
- cell 1: a `Tesla T4` table
- cell 4: `device: cuda`, loss decreasing, `new best model saved`
- cell 5: a per-step time and full-run estimates

**For the real training run later:** upload your real clean images to `data/train` and `data/val` here, run the full `train.py` command from the README, then download `checkpoints/best.pth` and put it in the project's `checkpoints/` folder on your PC.